## Sales Performance → Monitor business growth 

**Insights:** 
- Which months are peak vs slow periods 
- Which products/categories generate most revenue 
- Whether sales are growing, flat, or declining 

**Decisions:**
- Launch promotions in slow months 
- Allocate more marketing to high-performing categories 
- Adjust pricing/visibility for declining products

## 1. Load Silver tables for order items, orders, customers, products, categories, and reviews.
- These tables are the foundational data sources for building Gold-level analytics tables.
- Loading them enables downstream joins and aggregations to generate business insights on sales, customers, and product performance.

In [0]:
from pyspark.sql import functions as F

df_order_items = spark.read.table("hive_metastore.silver_ecommerce.silver_order_items_2")
df_orders = spark.read.table("hive_metastore.silver_ecommerce.silver_orders_2")
df_customers = spark.read.table("hive_metastore.silver_ecommerce.silver_customers_2")
df_products = spark.read.table("hive_metastore.silver_ecommerce.silver_products_2")
df_categories = spark.read.table("hive_metastore.silver_ecommerce.silver_categories_2")
df_reviews = spark.read.table("hive_metastore.silver_ecommerce.silver_reviews_2")

## 2. Build Gold Table: sales_performance_gold

This cell creates the `sales_performance_gold` table by joining Silver tables for order items, orders, products, and categories.

- Renames the order creation timestamp for clarity.
- Joins order items with orders, products, and categories to enrich each sales record.
- Extracts year and month from the order date for time-based analysis.
- Displays the resulting DataFrame for inspection.
- Saves the enriched sales performance data as a Gold Delta table for downstream analytics.


In [0]:
from pyspark.sql.functions import month, year

df_orders_renamed = df_orders.withColumnRenamed("created_at", "order_created_at")

df_sales_performance = (
    df_order_items
    .join(df_orders_renamed, "order_id")
    .join(df_products, "product_id")
    .join(df_categories, "category_id")
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
)

display(df_sales_performance)

df_sales_performance.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.gold_ecommerce.sales_performance_gold")

In [0]:
spark.read.table("hive_metastore.gold_ecommerce.sales_performance_gold").display()

##3. Customer Insights (customer_insights_gold)
## 



Insights:
•	Who your top spenders and loyal customers are

•	Which segments (Budget, Premium, Regular) bring the most revenue

•	Which regions contribute the most

•	Who is at risk of losing customers


Decisions:
•	Offer loyalty programs to Premium customers

•	Launch win-back campaigns for inactive users

•	Expand ads/operations in high-value regions

•	Focus customer service on top-spending clients


## What I did:
Joined orders → customers → order items.

Aggregated spending, last order date, and total orders per customer.

## Why:

To identify top spenders, loyal customers, segments, and regions.

Helps detect churn risk and plan loyalty programs.

## Columns:[](url)

customer_id, first_name, last_name → customer identifier

customer_segment → Budget, Premium, Regular

country, city → location

total_spent → revenue from this customer

last_order_date → last activity for churn detection

total_orders → loyalty indicator

In [0]:
from pyspark.sql import functions as F

df_customer_insights = (
    df_orders
    .join(df_customers, "customer_id")
    .join(df_order_items, "order_id")
    .groupBy(
        "customer_id", "first_name", "last_name",
        "customer_segment", "country", "city"
    )
    .agg(
        F.sum("line_total").alias("total_spent"),
        F.max("order_date").alias("last_order_date"),
        F.countDistinct("order_id").alias("total_orders")
    )
)

df_customer_insights.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.gold_ecommerce.customer_insights_gold")



In [0]:
spark.read.table("hive_metastore.gold_ecommerce.customer_insights_gold").display()

## 4. Build Gold Table: product_reviews_gold

This cell creates the `product_reviews_gold` table by enriching review data with product and customer information.

- Joins the products and categories tables to associate each product with its category.
- Enriches reviews by joining with product details (name, category) and customer attributes (segment, country).
- Filters out reviews that do not have a rating.
- Adds a sentiment label based on the rating: "Positive" (rating >= 4), "Neutral" (rating == 3), or "Negative" (rating < 3).
- Saves the resulting DataFrame as a Gold Delta table for downstream analytics.

In [0]:
df_products_categaories = (
    df_products.alias("p")
    .join(
        df_categories.select("category_id", "category_name").alias("c"),
        on="category_id",
        how="left"
    )
)

# Enrich reviews with product + customer info
df_reviews_enriched = (
    df_reviews.alias("r")
    .join(
        df_products_categaories.select("product_id", "product_name", "category_name").alias("p"),
        on="product_id",
        how="inner"  
    )
    .join(
        df_customers.select("customer_id", "customer_segment", "country").alias("c"),
        on="customer_id",
        how="left"
    )
    .filter(F.col("r.rating").isNotNull())
    .withColumn(
        "sentiment_label",
        F.when(F.col("rating") >= 4, "Positive")
         .when(F.col("rating") == 3, "Neutral")
         .otherwise("Negative")
    )
)
    
#display(df_reviews_enriched)

# Save as Gold table
df_reviews_enriched.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("gold_ecommerce.product_reviews_gold")